# Train Denoise ECG (haar_sym_lite) trên Google Colab

Notebook chạy đầy đủ: tải dữ liệu MIT-BIH + NSTDB, tiền xử lý, sweep **depth / wavelet / loss**, evaluate, và tự chỉ ra depth/loss/wavelet tốt nhất.

**Trước khi chạy:** `Runtime -> Change runtime type -> T4 GPU`. Sau đó `Runtime -> Run all`.

**Chuẩn bị 1 lần:** upload file zip code lên Google Drive (bất kỳ chỗ nào trong *Drive của tôi*). Cell mục 1 sẽ **tự tìm** file zip có chữ "Denoise" và **tự giải nén cả zip lồng nhau** — không cần sửa đường dẫn tay. Nếu có nhiều zip, nó chọn file lớn nhất.

## 0. Kiểm tra GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa bật GPU! Runtime -> Change runtime type -> T4 GPU rồi Run all lại.'

## 1. Mount Drive + giải nén code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # vẫn mount Drive để lưu runs/ (khỏi mất khi Colab ngắt)

import os, glob, shutil, zipfile

# Để TRỐNG để tự tìm zip; hoặc điền đường dẫn tuyệt đối nếu muốn.
ZIP_ON_DRIVE = ''

if not ZIP_ON_DRIVE:
    # Tìm ở /content trước (nếu bạn upload thẳng vào Colab), không có thì sang Drive.
    hits = glob.glob('/content/*Denoise*.zip')
    if not hits:
        hits = glob.glob('/content/drive/MyDrive/**/*Denoise*.zip', recursive=True)
    assert hits, 'Khong thay zip chua "Denoise". Upload vao Colab hoac Drive, hoac dien tay ZIP_ON_DRIVE.'
    ZIP_ON_DRIVE = max(hits, key=os.path.getsize)   # chon zip lon nhat = ban day du
print('Dung zip:', ZIP_ON_DRIVE)

shutil.rmtree('/content/proj', ignore_errors=True)
os.makedirs('/content/proj', exist_ok=True)
shutil.copy(ZIP_ON_DRIVE, '/content/proj/code.zip')

# Giai nen KE CA zip long trong zip
while True:
    zips = glob.glob('/content/proj/**/*.zip', recursive=True)
    if not zips:
        break
    for zp in zips:
        out = zp[:-4]
        os.makedirs(out, exist_ok=True)
        try:
            with zipfile.ZipFile(zp) as z:
                z.extractall(out)
        except zipfile.BadZipFile:
            pass
        os.remove(zp)

cands = [os.path.dirname(p) for p in glob.glob('/content/proj/**/configs/default.yaml', recursive=True)]
assert cands, 'Khong tim thay configs/default.yaml sau khi giai nen.'
PROJ = cands[0]
print('PROJECT DIR =', PROJ)
os.chdir(PROJ)

In [ ]:
import glob, os
req = glob.glob('/content/proj/**/requirements.txt', recursive=True)
if req:
    os.chdir(os.path.dirname(req[0]))   # về đúng thư mục code
    print('Cai tu:', req[0])
    !pip -q install -r requirements.txt
else:
    print('Khong thay requirements.txt -> cai truc tiep')
    !pip -q install numpy scipy pandas matplotlib pywavelets wfdb PyYAML tqdm torch torchvision

## 2. Tải dữ liệu PhysioNet (MITDB + NSTDB)
Chạy 1 lần. Nếu đã có backup processed trên Drive thì có thể bỏ qua cell tải + tiền xử lý và dùng cell 'Khôi phục' ở mục 4.

In [ ]:
import importlib
if importlib.util.find_spec('wfdb') is None:
    !pip -q install wfdb

import wfdb
wfdb.dl_database('mitdb', '/content/data/MITDB')   # ~48 record ECG sạch
wfdb.dl_database('nstdb', '/content/data/NSTDB')   # nhiễu bw / ma / em
print('Downloaded.')

## 3. Sửa config cho môi trường Colab (đường dẫn + nơi lưu runs)

In [ ]:
import yaml
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'   # lưu checkpoint trên Drive để không mất khi Colab ngắt

cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['dataset_root']   = '/content/data'
cfg['data']['processed_root'] = '/content/data/processed'
cfg['output_root']            = RUNS_DIR
cfg['train']['device']        = 'cuda'
# (Tùy chọn) giảm epoch cho nhanh nếu sợ hết 12h — early stopping vẫn bật:
# cfg['train']['epochs'] = 30
yaml.safe_dump(cfg, open('configs/default.yaml','w'), sort_keys=False)
print(open('configs/default.yaml').read())

## 4. Tiền xử lý (tạo manifest.csv + segments .npz) — chạy 1 lần

In [ ]:
!PYTHONPATH=. python -m src.prepare_data --config configs/default.yaml
# Sao lưu để lần sau khỏi làm lại:
!tar czf /content/drive/MyDrive/processed_backup.tar.gz -C /content/data processed
print('Prepared + backed up.')

In [ ]:
# (CHỈ dùng khi reconnect và ĐÃ có processed_backup.tar.gz) — khôi phục nhanh, khỏi tải + tiền xử lý lại:
# !mkdir -p /content/data && tar xzf /content/drive/MyDrive/processed_backup.tar.gz -C /content/data
# print('Restored processed.')

## 5. Train các SWEEP để trả lời 3 câu hỏi
Mỗi sweep giữ các yếu tố khác ở mặc định (base=32, exp=4). Có thể mất vài giờ — checkpoint lưu trên Drive nên ngắt giữa chừng vẫn giữ được run đã xong.

In [ ]:
# (1) SWEEP DEPTH: mid_depth = 1..5  (haar, mse)
!bash scripts/sweep_depth.sh

In [ ]:
# (2) SWEEP WAVELET: haar db2 db4 sym4 coif1 bior2.2  (mid=3, mse)
!bash scripts/sweep_wavelets.sh

In [ ]:
# (3) SWEEP LOSS: mixed alpha 0..1 (haar, mid=3) + 2 loss còn lại để so sánh đủ
!bash scripts/sweep_alpha.sh
!python -m src.train --config configs/default.yaml --model haar_sym_lite --wavelet haar --loss mse      --mid_depth 3
!python -m src.train --config configs/default.yaml --model haar_sym_lite --wavelet haar --loss mse_haar --mid_depth 3

## 6. Evaluate + tổng hợp

In [ ]:
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'
!python -m src.eval --config configs/default.yaml --run_glob "{RUNS_DIR}/*"
!python scripts/summarize_runs.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/summary_all_models.csv"

## 7. TRẢ LỜI: depth / wavelet / loss nào tốt nhất?
Tiêu chí: **snr_imp CAO** là tốt; **prd, rmse THẤP** là tốt. Dòng đầu mỗi bảng = winner.

In [ ]:
import pandas as pd
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'
s = pd.read_csv(f'{RUNS_DIR}/summary_all_models.csv')
hs = s[s.model_name=='haar_sym_lite'].copy()

print('==================== DEPTH tốt nhất ====================')
d = hs[(hs.loss=='mse') & (hs.wavelet=='haar')].sort_values('snr_imp', ascending=False)
print(d[['mid_depth','snr_imp','prd','rmse','cosine']].to_string(index=False))

print('\n==================== WAVELET tốt nhất ====================')
w = hs[(hs.loss=='mse') & (hs.mid_depth==3)].sort_values('snr_imp', ascending=False)
print(w[['wavelet','snr_imp','prd','rmse','cosine']].to_string(index=False))

print('\n==================== LOSS / ALPHA tốt nhất ====================')
l = hs[(hs.wavelet=='haar') & (hs.mid_depth==3)].sort_values('snr_imp', ascending=False)
print(l[['loss','alpha','snr_imp','prd','rmse','cosine']].to_string(index=False))

print('\n>>> Lấy giá trị ở dòng đầu 3 bảng để điền vào cell mục 8 (model tối ưu).')

## 8. Train MODEL TỐI ƯU (kết hợp best depth + best wavelet + best loss)
Sửa 4 biến dưới theo winner ở mục 7, rồi chạy.

In [ ]:
BEST_DEPTH   = 3        # <-- điền mid_depth tốt nhất
BEST_WAVELET = 'haar'  # <-- điền wavelet tốt nhất
BEST_LOSS    = 'mse'   # 'mse' | 'mixed' | 'mse_haar'
BEST_ALPHA   = 1.0     # chỉ dùng khi loss='mixed'

!python -m src.train --config configs/default.yaml --model haar_sym_lite \
    --wavelet {BEST_WAVELET} --loss {BEST_LOSS} --alpha {BEST_ALPHA} --mid_depth {BEST_DEPTH} --run_suffix optimal

## 9. (Tùy chọn) Train 5 baseline để so sánh cuối cùng

In [ ]:
!bash scripts/train_baselines.sh

In [ ]:
# Eval + tổng hợp lại TOÀN BỘ rồi xếp hạng model theo snr_imp
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'
!python -m src.eval --config configs/default.yaml --run_glob "{RUNS_DIR}/*"
!python scripts/summarize_runs.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/summary_all_models.csv"
import pandas as pd
s = pd.read_csv(f'{RUNS_DIR}/summary_all_models.csv').sort_values('snr_imp', ascending=False)
print(s[['run_name','model_name','wavelet','loss','alpha','mid_depth','snr_imp','prd','rmse','cosine']].to_string(index=False))

In [ ]:
# ===== 10. Bao cao do phuc tap model: so tham so + latency (GPU) =====
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'
!PYTHONPATH=. python scripts/report_model_complexity.py \
    --models dw_cnn dw_se dnn_dan fcn liwave haar_sym_lite \
    --wavelets haar --latency --out "{RUNS_DIR}/model_complexity_report.csv"
# Ghi chu: report tren haar_sym_lite mac dinh (base32/exp4/mid3). Neu muon dung dung
# cau hinh optimal, them: --mid_depth <BEST_DEPTH> --wavelets <BEST_WAVELET>

In [ ]:
# ===== 11. Ve hinh: bieu do metric + waveform truoc/sau khu nhieu =====
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'
# Bang tong hop theo noise & SNR (neu chua co)
!python scripts/merge_noise_snr_summary.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/all_models_by_noise_snr.csv"
# Bieu do so sanh model / theo SNR / theo noise
!PYTHONPATH=. python scripts/plot_results.py --summary "{RUNS_DIR}/summary_all_models.csv" --noise_snr "{RUNS_DIR}/all_models_by_noise_snr.csv" --out_dir "{RUNS_DIR}/figures/metrics"
# Waveform truoc/sau tren model TOI UU (luu vao trong thu muc run *optimal*)
!python -m src.visualize --config configs/default.yaml --run_glob "{RUNS_DIR}/*optimal*" --segment_index 0

# Hien thi vai hinh ngay trong notebook
from IPython.display import Image, display
import glob
imgs = sorted(glob.glob(f'{RUNS_DIR}/figures/metrics/*.png')) + sorted(glob.glob(f'{RUNS_DIR}/*optimal*/figures/waveforms/*.png'))
for p in imgs[:8]:
    print(p); display(Image(p))

In [ ]:
# ===== 12. Bang mean±std + kiem dinh thong ke (optimal vs baseline tot nhat) =====
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
RUNS_DIR = '/content/drive/MyDrive/denoise_runs'

# Gom metric TUNG segment cua moi run
!python scripts/merge_all_segment_csv.py --run_glob "{RUNS_DIR}/*" --out "{RUNS_DIR}/all_segment_metrics.csv"
seg = pd.read_csv(f'{RUNS_DIR}/all_segment_metrics.csv')

metrics = ['snr_imp', 'prd', 'rmse', 'mae', 'cosine']
def mean_std(g):
    return pd.Series({m: f"{g[m].mean():.4f} ± {g[m].std():.4f}" for m in metrics})
tbl = seg.groupby('run_name').apply(mean_std)
print('===== MEAN ± STD theo run ====='); print(tbl.to_string())

# Chon run 'optimal' va baseline tot nhat theo snr_imp trung binh
mean_snr = seg.groupby('run_name')['snr_imp'].mean()
opt_cands = [r for r in mean_snr.index if 'optimal' in r]
opt_run = opt_cands[0] if opt_cands else mean_snr.idxmax()
baselines = ['dw_cnn', 'dw_se', 'dnn_dan', 'fcn', 'liwave']
base_runs = seg[seg.model_name.isin(baselines)].groupby('run_name')['snr_imp'].mean()
best_base_run = base_runs.idxmax()
print(f"\nOptimal    : {opt_run}\nBest base  : {best_base_run}")

# Ghep theo tung segment (test set giong nhau) roi test cap doi Wilcoxon
key = ['record_id', 'segment_id', 'noise_type', 'noise_snr']
a = seg[seg.run_name == opt_run].set_index(key)['snr_imp']
b = seg[seg.run_name == best_base_run].set_index(key)['snr_imp']
common = a.index.intersection(b.index)
stat, p = wilcoxon(a.loc[common].values, b.loc[common].values)
print(f"\nWilcoxon snr_imp (optimal vs best baseline): n={len(common)}, p-value={p:.3e}")
print('=> p<0.05: cai thien CO y nghia thong ke' if p < 0.05 else '=> chua du y nghia thong ke')